# Main-text Figure 4

This notebook reads only the final plot-ready baseline-risk and monthly NPPeco-anomaly arrays.

In [ ]:
import json
import shutil
import tempfile
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import xarray as xr
import ultraplot as uplt
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from shapely.geometry import shape

MAP_PATH = Path("../data/main_figures/figure4/maps.nc")
WORLD_PATH = Path("../data/main_figures/world_simplified.geojson")
OUTPUT = Path("../outputs/main_figures")
OUTPUT.mkdir(parents=True, exist_ok=True)

with WORLD_PATH.open(encoding="utf-8") as stream:
    world = json.load(stream)
LAND = tuple(shape(feature["geometry"]) for feature in world["features"])


@contextmanager
def open_dataset_portable(path):
    path = Path(path)
    temporary = None
    try:
        try:
            dataset = xr.open_dataset(path)
        except (FileNotFoundError, OSError):
            with tempfile.NamedTemporaryFile(prefix="hanpp_plot_", suffix=".nc", delete=False) as handle:
                temporary = Path(handle.name)
            shutil.copy2(path, temporary)
            dataset = xr.open_dataset(temporary)
        try:
            yield dataset
        finally:
            dataset.close()
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)


def show_map(ax, values, cmap, vmin, vmax):
    image = ax.imshow(
        values,
        origin="lower",
        extent=(-180, 180, -60, 80),
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        interpolation="nearest",
        regrid_shape=900,
    )
    ax.set_extent((-180, 180, -60, 80), crs=ccrs.PlateCarree())
    # ax.add_geometries(
    #     LAND,
    #     crs=ccrs.PlateCarree(),
    #     facecolor="none",
    #     edgecolor="0.25",
    #     linewidth=0.25,
    #     zorder=3,
    # )
    return image


with open_dataset_portable(MAP_PATH) as dataset:
    maps = dataset.load()


In [ ]:
layout = [
    [1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3],
    [4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 7],
    [8, 8, 8, 9, 9, 9, 10, 10, 10, 11, 11, 11],
    [12, 12, 12, 13, 13, 13, 14, 14, 14, 15, 15, 15],
]

fig, axs = uplt.subplots(
    layout,
    proj="robin",
    figwidth="20cm",
    dpi=100,
    share=False,
    span=False,
    refnum=4,
    hratios=(1.30, 1, 1, 1),
    wspace=0.25,
    hspace=(1.3, 0.7, 0.7),
    right="4.5em",
)

for ax, taxon in zip(axs[:3], maps["taxon"].to_numpy().tolist()):
    # land outside the taxon hotspot is shown in grey (not assessed)
    ax.add_geometries(LAND, crs=ccrs.PlateCarree(), facecolor="0.82", edgecolor="none", zorder=0)
    show_map(
        ax,
        maps["baseline_risk"].sel(taxon=taxon).to_numpy(),
        cmap="turbo",
        vmin=0,
        vmax=1,
    )

for ax, month in zip(axs[3:], range(1, 13)):
    show_map(
        ax,
        maps["NPPeco_anomaly"].sel(month=month).to_numpy(),
        cmap="RdYlGn_r",
        vmin=-3,
        vmax=3,
    )

axs.format(
    abc=True,
    abcloc="ul",
    coast=True,
    lw=0.5,
    grid=True,
    lonlim=(-180, 180),
    latlim=(-60, 90),
)
axs[:3].format(title=("mammal", "amphibian", "reptile"))
axs[3:].format(title="")

risk_mappable = plt.cm.ScalarMappable(
    norm=plt.Normalize(vmin=0, vmax=1),
    cmap="turbo",
)
risk_mappable.set_array([])
fig.colorbar(
    risk_mappable,
    loc="r",
    row=1,
    width="1em",
    label="Pressure index",
    ticks=np.linspace(0, 1, 5),
)

anomaly_mappable = plt.cm.ScalarMappable(
    norm=plt.Normalize(vmin=-3, vmax=3),
    cmap="RdYlGn_r",
)
anomaly_mappable.set_array([])
fig.colorbar(
    anomaly_mappable,
    loc="r",
    rows=(2, 4),
    width="1em",
    label=r"$\mathrm{NPP_{eco}}$ anomaly ($-Z$)",
    ticks=np.arange(-3, 4, 1),
    extend="both",
)

fig.savefig(OUTPUT / "Fig4.png", dpi=300, bbox_inches=None, facecolor="white")
fig
